# Merge Nextflow Per-Sample Outputs

**Genesis Workbench** — Single Cell Module

This notebook runs as the final task in a parallel fan-out Nextflow job. It:
1. Scans per-sample output directories for h5ad / count matrix files
2. Optionally concatenates all samples into a single merged h5ad
3. Reports a summary of all samples
4. Sets `h5ad_paths` task value for downstream scanpy jobs

In [0]:
dbutils.widgets.text("output_dir", "", "Base Output Directory")
dbutils.widgets.text("sample_names", "", "Sample Names (comma-separated)")
dbutils.widgets.dropdown("merge_mode", "list_only", ["list_only", "concatenate"], "Merge Mode")

output_dir = dbutils.widgets.get("output_dir")
sample_names = [s.strip() for s in dbutils.widgets.get("sample_names").split(",") if s.strip()]
merge_mode = dbutils.widgets.get("merge_mode")

print(f"Output dir:    {output_dir}")
print(f"Samples ({len(sample_names)}): {sample_names}")
print(f"Merge mode:    {merge_mode}")

In [0]:
import os
import glob

results = []
all_h5ad = []

for sample in sample_names:
    sample_out = os.path.join(output_dir, "per_sample", sample, "output")
    
    # Look for h5ad files
    h5ad_files = glob.glob(os.path.join(sample_out, "**/*.h5ad"), recursive=True)
    
    # Also look for filtered count matrices (as fallback)
    mtx_files = glob.glob(os.path.join(sample_out, "**/filtered**/matrix.mtx*"), recursive=True)
    
    status = "found" if h5ad_files else ("mtx_only" if mtx_files else "missing")
    
    results.append({
        "sample": sample,
        "status": status,
        "h5ad_files": h5ad_files,
        "mtx_dirs": [os.path.dirname(f) for f in mtx_files],
        "output_dir": sample_out,
    })
    
    all_h5ad.extend(h5ad_files)

print(f"\n{'Sample':<30} {'Status':<12} {'h5ad':<5} {'MTX dirs':<5}")
print("=" * 60)
for r in results:
    print(f"{r['sample']:<30} {r['status']:<12} {len(r['h5ad_files']):<5} {len(r['mtx_dirs']):<5}")
    for h in r['h5ad_files']:
        print(f"  -> {h}")

print(f"\nTotal h5ad files found: {len(all_h5ad)}")

In [0]:
import sys
sys.path.insert(0, "/Workspace/Users/andrew_forman@eisai.com/.bundle/genesis_workbench/core/prod/files/app")
from utils.nextflow_pipeline import convert_mtx_to_h5ad

converted = []
for r in results:
    if r['status'] == 'mtx_only':
        for mtx_dir in r['mtx_dirs']:
            h5ad_path = os.path.join(r['output_dir'], f"{r['sample']}_filtered.h5ad")
            conv = convert_mtx_to_h5ad(mtx_dir, h5ad_path)
            if conv['status'] == 'success':
                print(f"  Converted {r['sample']}: {conv['n_cells']:,} cells x {conv['n_genes']:,} genes")
                all_h5ad.append(h5ad_path)
                converted.append(h5ad_path)
            else:
                print(f"  Failed {r['sample']}: {conv['error']}")

if converted:
    print(f"\nConverted {len(converted)} MTX dirs to h5ad")
else:
    print("No MTX conversions needed")

In [0]:
if merge_mode == "concatenate" and len(all_h5ad) > 1:
    try:
        import anndata as ad
        print(f"Concatenating {len(all_h5ad)} h5ad files...")
        
        adatas = {}
        for h5ad_path in all_h5ad:
            # Extract sample name from path
            parts = h5ad_path.split("/per_sample/")
            sample_key = parts[1].split("/")[0] if len(parts) > 1 else os.path.basename(h5ad_path)
            adatas[sample_key] = ad.read_h5ad(h5ad_path)
            print(f"  Loaded {sample_key}: {adatas[sample_key].n_obs:,} cells")
        
        merged = ad.concat(adatas, label="sample", keys=list(adatas.keys()), join="outer")
        merged_path = os.path.join(output_dir, "merged_all_samples.h5ad")
        merged.write_h5ad(merged_path)
        
        print(f"\nMerged: {merged.n_obs:,} cells x {merged.n_vars:,} genes")
        print(f"Saved to: {merged_path}")
        
        # Set the merged path as the primary output
        dbutils.jobs.taskValues.set(key="h5ad_path", value=merged_path)
        dbutils.jobs.taskValues.set(key="h5ad_paths", value=merged_path)
        
    except ImportError:
        print("anndata not available — falling back to list mode")
        merge_mode = "list_only"
    except Exception as e:
        print(f"Concatenation failed: {e} — falling back to list mode")
        merge_mode = "list_only"

if merge_mode == "list_only" or len(all_h5ad) == 1:
    paths_str = ",".join(all_h5ad)
    dbutils.jobs.taskValues.set(key="h5ad_paths", value=paths_str)
    if all_h5ad:
        dbutils.jobs.taskValues.set(key="h5ad_path", value=all_h5ad[0])
    print(f"\nTask values set:")
    print(f"  h5ad_paths = {paths_str}")

In [0]:
found = sum(1 for r in results if r['status'] != 'missing')
missing = sum(1 for r in results if r['status'] == 'missing')

print("\n" + "=" * 60)
print(f"  Parallel Nextflow Fan-Out — Merge Summary")
print("=" * 60)
print(f"  Total samples:  {len(sample_names)}")
print(f"  Outputs found:  {found}")
if missing:
    print(f"  Missing:        {missing}")
    for r in results:
        if r['status'] == 'missing':
            print(f"    - {r['sample']}")
print(f"  h5ad files:     {len(all_h5ad)}")
print(f"  Merge mode:     {merge_mode}")
print("=" * 60)

if all_h5ad:
    print(f"\nReady for scanpy! Use the paths above in the Run Analysis tab.")
    for p in all_h5ad:
        print(f"  {p}")
else:
    print("\nNo h5ad files found. Check the individual Nextflow task logs.")